# Silver — GDP Indicators
Cleans and enriches Bronze GDP data: adds a proper date column for each quarter and rounds values.

**Source:** `bronze.statistics_iceland_raw`  
**Output:** `silver.gdp_indicators` (Delta table)  
**Date rule:** Q1 → Jan 1, Q2 → Apr 1, Q3 → Jul 1, Q4 → Oct 1

In [ ]:
df_raw = spark.read.table("bronze.statistics_iceland_raw")
df_raw.createOrReplaceTempView("v_gdp_raw")

In [ ]:
df_silver = spark.sql("""
    WITH raw_parsed AS (
        SELECT
            CAST(SUBSTRING(quarter_raw, 1, 4) AS INT) AS year,
            CAST(SUBSTRING(quarter_raw, 6, 1) AS INT) AS q,
            CAST(value_raw AS DOUBLE) AS value,
            quarter_raw,
            ingested_at
        FROM v_gdp_raw
    ),
    date_constructed AS (
        SELECT
            year,
            q,
            value,
            quarter_raw,
            ingested_at,
            TO_DATE(CONCAT_WS('-', year, (q * 3 - 2), '1'), 'yyyy-M-d') AS date,
            'gdp_yoy_growth' AS metric
        FROM raw_parsed
        WHERE year IS NOT NULL
        AND q IS NOT NULL
    ),
    deduplicated AS (
        SELECT 
            quarter_raw,
            year,
            q,
            date,
            metric,
            value,
            ingested_at,
            ROW_NUMBER() OVER (PARTITION BY year, q, metric ORDER BY ingested_at DESC) as rn
        FROM date_constructed
    )
    SELECT 
        quarter_raw AS quarter, 
        year, 
        q, 
        date, 
        metric, 
        value, 
        CURRENT_TIMESTAMP() AS processed_at
    FROM deduplicated
    WHERE rn = 1
""")

df_silver.createOrReplaceTempView("v_gdp_silver_staged")

if df_silver.isEmpty():
    print("No valid GDP data found after cleaning. Exiting.")
    mssparkutils.notebook.exit("no-data")

In [ ]:
if not spark.catalog.tableExists("silver.gdp_indicators"):
    df_silver.write.format("delta").saveAsTable("silver.gdp_indicators")
    print("Created silver.gdp_indicators.")
else:
    spark.sql("""
        MERGE INTO silver.gdp_indicators AS target
        USING v_gdp_silver_staged AS source
        ON target.year = source.year 
        AND target.q = source.q 
        AND target.metric = source.metric
        WHEN MATCHED THEN UPDATE SET
            target.quarter      = source.quarter,
            target.value        = source.value,
            target.date         = source.date,
            target.processed_at = source.processed_at
        WHEN NOT MATCHED THEN INSERT (quarter, year, q, date, metric, value, processed_at)
        VALUES (source.quarter, source.year, source.q, source.date, source.metric, source.value, source.processed_at)
    """)

print(f"Merge complete. Final count: {spark.table('silver.gdp_indicators').count()}")